# Obtaining Uncalibrated JWST Data

In this tutorial we will use ```astropy.fits.open``` and/or ```jwst_mast_query``` to obtain ```uncal``` JWST data files which are needed for the BREADS analysis.

### Downloading public data (Easy)

If you know the exact filenames you would like to download and the datasets have completed their proprietary period to become public, it is straightforward to obtain them directly through the MAST API using ```astropy.fits.open```.

In [2]:
import astropy.io.fits as fits
import os

**This cell must be modified!** Specify the directory you would like to store data products in

In [3]:
base_path = '/astro/epsig/tutorial/'

These subdirectories will be useful for organizing data products.

In [4]:
data_path = base_path+'data/'
raw_path = data_path+'raw/'
create_paths = [base_path,data_path,raw_path]
for path in create_paths:
    if not os.path.exists(path):
        os.mkdir(path)

The next cell only downloads 4 files (2 for nrs1 and 2 for nrs2) for testing purposes. See next

In [5]:
filelist = ['jw01414013001_02101_00001_nrs1_uncal.fits',
            'jw01414013001_02101_00001_nrs2_uncal.fits',
            'jw01414013001_02101_00002_nrs1_uncal.fits',
            'jw01414013001_02101_00002_nrs2_uncal.fits']
for file in filelist:
    print('downloading {} -> {}'.format(file,raw_path))
    mast_file_url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/{file}"
    hdul = fits.open(mast_file_url)
    hdul.writeto(raw_path+file,overwrite=True)

downloading jw01414013001_02101_00001_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414013001_02101_00001_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414013001_02101_00002_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414013001_02101_00002_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/


This loop iterates over a predefined sequence of observations from GTO program 1414, specifically sequences 12, 13, and 14, observations numbered 1-9, and both nrs1 and nrs2 detectors. The resulting files are downloaded and saved under the "raw" subdirectory.

In [5]:
for seq_num in ['012','013','014']:
    for obs_num in [str(x) for x in range(1,9+1)]:
        for det_string in ['nrs1','nrs2']:
            file = 'jw01414'+seq_num+'001_02101_0000'+obs_num+'_'+det_string+'_uncal.fits'
            print('downloading {} -> {}'.format(file,raw_path))
            mast_file_url = f"https://mast.stsci.edu/api/v0.1/Download/file?uri=mast:JWST/product/{file}"
            hdul = fits.open(mast_file_url)
            hdul.writeto(raw_path+file,overwrite=True)

downloading jw01414012001_02101_00001_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00001_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00002_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00002_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00003_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00003_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00004_nrs1_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/
downloading jw01414012001_02101_00004_nrs2_uncal.fits -> /stow/jruffio/data/JWST/tutorial/data/raw/


KeyboardInterrupt: 

Thats it! If you just want to run the tutorial series, you can move onto notebook #2. If you are looking to download exclusive access data which is not public, the process is slightly more involved, but explained below.

### Downloading exclusive access data with jwst_mast_query (Medium)

In order to obtain uncalibrated data files for observations which are not yet public, it is required (to my knowledge) to first install ```jwst_mast_query``` (https://github.com/spacetelescope/jwst_mast_query). Additionally, one must have acess to a MAST account with the proper exclusive access dataset authorization (ask your PI for help.) Once this is in order, you can genereate a specific exclusive access token at (https://auth.mast.stsci.edu/token) to properly communicate your privileged status.

**This cell must be modified!** Replace the #### with your token.

In [ ]:
token = '####'

os.environ["MAST_API_TOKEN"] = token

This cell will check your token is properly set as an environment varialbe.

In [ ]:
import subprocess
return_token = subprocess.check_output('echo $MAST_API_TOKEN',shell=True)
assert token == str(return_token)[2:-3]

**This cell must be modified!** Replace 'n' with 'y' to proceed with the download. Otherwise it will simply print a list of files matching the search criteria.

In [ ]:
DL = 'n'

**This cell must be modified!** Replace the path to your installation of ```jwst_mast_query```, specifically the directory where you can find ```jwst_dowload.py```

In [ ]:
JWSTDL_path = '/home/amadurowicz/miniconda3/envs/stenv/bin/'

The following cell uses the shell to call ```jwst_download.py``` with specific arguments for our purpose. You can modify the ```propID``` and ```date_string``` to select other sequences of observations. The current configuration will download the tutorial dataset from GTO 1414 on HD 19467 B.

In [ ]:
def download(target):
    
    match target:
        case 'HD 19467':
            date_string = '2024-01-23 2024-01-26'
    
    cmd = """\
    while true; do echo {}; sleep 1; done | \
    '{}jwst_download.py' \
    --propID 1414 \
    --instrument nirspec \
    --filetypes 'uncal' \
    --outrootdir '{}' \
    --outsubdir 'data/raw' \
    --skip_propID2outsubdir \
    --date_select {} \
    """.format(DL, JWSTDL_path, base_path, date_string)
    
    subprocess.call(cmd,shell=True)

download('HD 19467')